In [1]:
import os
import numpy as np
import pydicom as dicom

In [3]:
import pydicom

def extract_position_check(dcm_path, slice_index=0):
    """
    Extracts specific Siemens private tags from a given DICOM file.

    Parameters
    ----------
    dcm_path : str
        Path to the DICOM file.
    slice_index : int
        Index of the slice in the Per-frame Functional Groups Sequence.

    Returns
    -------
    tuple
        (value_1188, last_value_1159) corresponding to:
        dcm[0x52009230][slice_index][0x002111fe][0][0x00211188].value
        dcm[0x52009230][slice_index][0x002111fe][0][0x00211159].value[-1]
    """
    # Load DICOM
    dcm = pydicom.dcmread(dcm_path)

    # Access Per-frame Functional Groups Sequence
    pf_seq = dcm[0x52009230].value

    # Select slice index
    pf_item = pf_seq[slice_index]

    # Access private sequence (0021,11fe)
    private_seq = pf_item[0x002111fe].value

    # Extract desired values
    value_1188 = private_seq[0][0x00211188].value
    print(private_seq[0][0x00211159].value)
    last_value_1159 = private_seq[0][0x00211159].value[-1]

    return value_1188, last_value_1159


In [14]:
root_dir = "/Volumes/T7/Stim-CODE/Dicoms"  # contains volunteer folders
target_scan_substr = "trap_mmt0_Ncalib2"

In [15]:
import os
import glob



results = {}

for volunteer in sorted(os.listdir(root_dir)):
    volunteer_path = os.path.join(root_dir, volunteer)
    if not os.path.isdir(volunteer_path):
        continue

    print(f"\nProcessing volunteer: {volunteer}")
    results[volunteer] = {}

    # head library level
    head_lib_folders = sorted(os.listdir(volunteer_path))

    for head_lib in head_lib_folders:
        head_lib_path = os.path.join(volunteer_path, head_lib)
        if not os.path.isdir(head_lib_path):
            continue

        # scan folder level: match substring
        scan_folders = sorted(os.listdir(head_lib_path))
        scan_path = None
        for scan in scan_folders:
            if target_scan_substr in scan:
                scan_path = os.path.join(head_lib_path, scan)
                break

        if scan_path is None or not os.path.isdir(scan_path):
            print(f"  ✗ No matching scan found in {head_lib_path}")
            continue

        # find one DICOM
        dcm_files = glob.glob(os.path.join(scan_path, '*.dcm'))
        if len(dcm_files) == 0:
            print(f"  ✗ No DICOMs found in {scan_path}")
            continue

        dcm_path = dcm_files[0]

        try:
            value_1188, last_value_1159 = extract_position_check(
                dcm_path,
                slice_index=15
            )

            results[volunteer][scan] = {
                '0021_1188': value_1188,
                '0021_1159_last': last_value_1159
            }

            # print both values
            print(f"  ✓ {scan}")
            print(f"    center slice: {value_1188}, isocenter: {last_value_1159}, diff: {last_value_1159 - round(value_1188)}")

        except Exception as e:
            print(f"  ✗ {scan} — {e}")


Processing volunteer: Sitm_CODE_V005_Sitm_CODE_V005_19951215
[0, 0, 6]
  ✓ trap_mmt0_Ncalib2_5_MR
    center slice: 5.68182, isocenter: 6, diff: 0

Processing volunteer: Stim_CODE_V001a_Stim_CODE_V001a_19991209
[0, 0, 2]
  ✓ trap_mmt0_Ncalib2_11_MR
    center slice: 2.3771, isocenter: 2, diff: 0

Processing volunteer: Stim_CODE_V002_Stim_CODE_V002_20001212
[0, 0, 13]
  ✓ trap_mmt0_Ncalib2_11_MR
    center slice: 12.8822, isocenter: 13, diff: 0

Processing volunteer: Stim_CODE_V003_Stim_CODE_V003_19951214
[0, 0, 7]
  ✓ trap_mmt0_Ncalib2_14_MR
    center slice: 6.81818, isocenter: 7, diff: 0

Processing volunteer: Stim_CODE_V004_Stim_CODE_V004_20011215
[0, 0, 6]
  ✓ trap_mmt0_Ncalib2_5_MR
    center slice: 5.68182, isocenter: 6, diff: 0

Processing volunteer: Stim_CODE_V006_Stim_CODE_V006_19921216
[0, 0, 8]
  ✓ trap_mmt0_Ncalib2_5_MR
    center slice: 7.95454, isocenter: 8, diff: 0

Processing volunteer: Stim_CODE_V007_Stim_CODE_V007_19921216
[0, 0, 13]
  ✓ trap_mmt0_Ncalib2_8_MR
    ce